In [ ]:
"""
Q-Learning LQR — LS + GD (Algoritmo B.1)
Transcripción fiel del código Octave — Apéndice B.

Notas de implementación:
- delta = 10^-2 = 0.01 según Tabla 6, PERO el código Octave usa
  con_error_k = 10e-2 = 0.1. Se usa 0.1 para reproducir la convergencia.
- El LS usa np.linalg.lstsq (equivalente numérico estable a inv(C_)*V*r
  de Octave cuando C_ es singular, como ocurre con solo 7 muestras).
- W_target en la tesis son los elementos de H_opt leídos fila por fila,
  NO el vector W con factores x2 de los off-diagonal.
- La convergencia correcta se verifica por K → K_opt ≈ [-1.17, 0.68].
"""
import sys
import numpy as np
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QHBoxLayout, QLabel, QPushButton, QSizePolicy
)
from PyQt6.QtCore import QTimer, Qt
from PyQt6.QtGui import QFont
import pyqtgraph as pg


# ─────────────────────────────────────────────────────────────
# michirp — idéntica al .m de referencia
# ─────────────────────────────────────────────────────────────
def sat(x):
    return np.clip(x, -1.0, 1.0)

def michirp(wini, wfin, N, T):
    u = np.zeros((N, 2))
    for i in range(1, N + 1):
        wk    = wini + (i - 1) / N * (wfin - wini)
        w_env = sat(10.0 * i / N) * sat((N - i) / (0.1 * N))
        u[i - 1, 0] = i * T
        u[i - 1, 1] = w_env * np.sin(wk * i * T)
    return u


# ─────────────────────────────────────────────────────────────
# Ventana principal
# ─────────────────────────────────────────────────────────────
class LQR_RL_App(QMainWindow):

    # ── Parámetros exactos del algoritmo ──────────────────────────────────
    A   = np.array([[1.8980, -0.9048], [1.0, 0.0]])
    B   = np.array([[1.0], [0.0]])
    Q_m = np.diag([1.0, 1.0])
    R_v = 2.0
    N_MAX   = 4000
    GAMMA   = 0.999
    N_LS    = 7        # tamaño lote LS
    ALPHA   = 120.02   # tasa aprendizaje GD
    CR      = 0.202    # escala chirp (Octave: b*inp(i,2)*0.202, b=1)
    # con_error_k = 10e-2 en Octave = 0.1
    DELTA   = 10e-2    # = 0.1  (Octave línea 54)

    # H inicial (Octave línea 24)
    H0 = np.array([[14., -2.,  2.],
                   [-8.,  3., -1.],
                   [ 8., -5.,  4.]], dtype=float)

    # H_opt (elementos, para líneas de referencia en gráfica de W)
    # Son los elementos de la fila de H_opt: H11, H12, H13, H22, H23, H33
    W_REF = [17.2962, -8.6202, 9.5272, 6.0227, -5.5512, 8.1353]

    def __init__(self):
        super().__init__()
        self.setWindowTitle("Q-Learning LQR — LS + GD  (Algoritmo B.1)")
        self.setStyleSheet("background:#0d0f14; color:#e0e0e0;")

        self.inp = michirp(0.005, 3_450_000, self.N_MAX + 1, 0.1)
        self._reset_state()
        self._build_ui()

        self.timer = QTimer()
        self.timer.timeout.connect(self._tick)
        self.timer.start(0)

    # ── Estado ────────────────────────────────────────────────────────────
    def _reset_state(self):
        self.H       = self.H0.copy()
        self.K       = self._K_from_H(self.H)
        self.x       = np.array([[5.0], [-4.0]])
        self.W_H_ant = np.zeros((6, 1))

        self.phi_ls  = np.zeros((6, self.N_LS))
        self.phi1_ls = np.zeros((6, self.N_LS))
        self.r_ls    = np.zeros((self.N_LS, 1))

        self.i         = 1
        self.n_updates = 0

        self.w_hist  = [[] for _ in range(6)]
        self.x1_hist = []

    @staticmethod
    def _K_from_H(H):
        return -(1.0 / H[2, 2]) * H[2, 0:2].reshape(1, 2)

    # ── GUI ───────────────────────────────────────────────────────────────
    def _build_ui(self):
        pg.setConfigOptions(antialias=True,
                            background="#000000",
                            foreground='#c0c0c0')
        cw = QWidget()
        self.setCentralWidget(cw)
        root = QHBoxLayout(cw)
        root.setSpacing(6); root.setContentsMargins(6, 6, 6, 6)

        # ── izquierda: gráfica W ──────────────────────────────────────────
        left = QVBoxLayout(); left.setSpacing(3)

        hdr_w = QLabel("Convergencia de W  (elementos de H)")
        hdr_w.setFont(QFont("Courier New", 9))
        hdr_w.setStyleSheet("color:#88ccff;")
        hdr_w.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        left.addWidget(hdr_w)

        self.plot_w = pg.PlotWidget()
        self.plot_w.setLabel('left', 'W'); self.plot_w.setLabel('bottom', 'iteración')
        self.plot_w.addLegend(offset=(5, 5))
        self.plot_w.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)

        COLS  = ['#ff6b6b','#ffd93d','#6bcb77','#4d96ff',"#9b25f5",'#ff9a3c']
        NAMES = ['H11','H12','H13','H22','H23','H33']
        self.w_curves = []
        for j in range(6):
            c = self.plot_w.plot([], pen=pg.mkPen(COLS[j], width=1.5), name=NAMES[j])
            self.w_curves.append(c)
            self.plot_w.addItem(pg.InfiniteLine(
                pos=self.W_REF[j], angle=0,
                pen=pg.mkPen(COLS[j], style=Qt.PenStyle.DashLine, width=1)
            ))
        left.addWidget(self.plot_w)
        root.addLayout(left, 2)

        # ── derecha: x1 + info ───────────────────────────────────────────
        right = QVBoxLayout(); right.setSpacing(3)

        hdr_x = QLabel("Estado  x₁  (posición)")
        hdr_x.setFont(QFont("Courier New", 9))
        hdr_x.setStyleSheet("color:#88ccff;")
        hdr_x.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        right.addWidget(hdr_x)

        self.plot_x = pg.PlotWidget()
        self.plot_x.setLabel('left', 'x₁'); self.plot_x.setLabel('bottom', 'muestra')
        self.plot_x.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)
        self.curve_x = self.plot_x.plot([], pen=pg.mkPen('#ffd93d', width=1.5))
        right.addWidget(self.plot_x)

        self.lbl = QLabel("Iniciando…")
        self.lbl.setFont(QFont("Courier New", 9))
        self.lbl.setStyleSheet(
            "background:#080b14; color:#00ff88; padding:7px; border:1px solid #1a2a4a;")
        self.lbl.setAlignment(Qt.AlignmentFlag.AlignLeft | Qt.AlignmentFlag.AlignTop)
        self.lbl.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        right.addWidget(self.lbl)

        btn = QPushButton("⟳  REINICIAR")
        btn.setStyleSheet(
            "background:#1a2a4a; color:#88ccff; font-size:11px; "
            "padding:5px; border:1px solid #2a3a6a; border-radius:3px;")
        btn.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        btn.clicked.connect(self._on_reset)
        right.addWidget(btn)

        root.addLayout(right, 1)

    def _on_reset(self):
        self._reset_state()
        for c in self.w_curves: c.setData([], [])
        self.curve_x.setData([], [])
        self.lbl.setText("Reiniciado.")
        if not self.timer.isActive(): self.timer.start(0)

    # ── Tick: 100 iteraciones por evento para GUI responsiva ──────────────
    def _tick(self):
        for _ in range(100):
            if self.i > self.N_MAX:
                self.timer.stop()
                break
            self._iteration()
        self._refresh_ui()

    # ── Una iteración del bucle `for i = 1:N` de Octave ──────────────────
    def _iteration(self):
        i  = self.i
        xi = self.x.copy()

        # control (Octave línea 73)
        u = float((self.K @ xi).item()) + float(self.inp[i - 1, 1]) * self.CR

        # dinámica (Octave línea 74)
        xn = self.A @ xi + self.B * u

        # u2 sin excitación para el siguiente estado (Octave línea 76)
        u2 = float((self.K @ xn).item())

        # buffer circular LS (Octave líneas 76-82)
        self.r_ls    = np.roll(self.r_ls,    -1, axis=0)
        self.phi_ls  = np.roll(self.phi_ls,  -1, axis=1)
        self.phi1_ls = np.roll(self.phi1_ls, -1, axis=1)

        self.r_ls[-1, 0] = float((xi.T @ self.Q_m @ xi).item()) + u**2 * self.R_v
        self.phi_ls[:,  -1] = [xi[0,0]**2, xi[0,0]*xi[1,0], xi[0,0]*u,
                               xi[1,0]**2, xi[1,0]*u,        u**2]
        self.phi1_ls[:, -1] = [xn[0,0]**2, xn[0,0]*xn[1,0], xn[0,0]*u2,
                               xn[1,0]**2, xn[1,0]*u2,       u2**2]

        # LS — inicialización en i == n_ls (Octave líneas 84-89)
        if i == self.N_LS:
            V = self.phi_ls - self.phi1_ls          # (6 x n_ls)
            # lstsq es el equivalente numérico estable de inv(V*V')*V*r
            # cuando V*V' es singular (rango 5 con solo 7 muestras)
            self.W_H_ant, _, _, _ = np.linalg.lstsq(V.T, self.r_ls, rcond=None)

        # GD — a partir de i > n_ls (Octave líneas 91-131)
        if i > self.N_LS:
            rk = float((xi.T @ self.Q_m @ xi).item()) + u**2 * self.R_v

            phi_k  = np.array([xi[0,0]**2, xi[0,0]*xi[1,0], xi[0,0]*u,
                               xi[1,0]**2, xi[1,0]*u,        u**2]).reshape(6, 1)
            phi1_k = np.array([xn[0,0]**2, xn[0,0]*xn[1,0], xn[0,0]*u2,
                               xn[1,0]**2, xn[1,0]*u2,       u2**2]).reshape(6, 1)

            vec_act = phi_k - self.GAMMA * phi1_k

            # paso GD (Octave línea 97)
            W_H = self.W_H_ant - self.ALPHA * vec_act * (
                float((self.W_H_ant.T @ vec_act).item()) - rk
            )

            # error TD (Octave línea 98)
            e_d_t = -(float((W_H.T @ vec_act).item())) + rk

            # actualizar H y K solo si pasa la condición (Octave línea 99)
            if abs(e_d_t) < self.DELTA:
                self.n_updates += 1
                W = W_H.flatten()
                self.H = np.array([[W[0],    W[1]/2,  W[2]/2],
                                   [W[1]/2,  W[3],    W[4]/2],
                                   [W[2]/2,  W[4]/2,  W[5]  ]])
                self.K       = self._K_from_H(self.H)
                self.W_H_ant = W_H.copy()

        # historial
        for j in range(6):
            self.w_hist[j].append(float(self.W_H_ant[j, 0]))
        self.x1_hist.append(float(xn[0, 0]))

        self.x = xn
        self.i += 1

    # ── Actualizar gráficas y texto ───────────────────────────────────────
    def _refresh_ui(self):
        n = len(self.w_hist[0])
        if n == 0:
            return
        xs = list(range(n))
        for j in range(6):
            self.w_curves[j].setData(xs, self.w_hist[j])
        self.curve_x.setData(self.x1_hist[-800:])

        W  = self.W_H_ant.flatten()
        Ke = -self.K
        estado = "✓ TERMINADO" if self.i > self.N_MAX else f"iteración {self.i-1}/{self.N_MAX}"
        self.lbl.setText(
            f"  {estado}   actualizaciones: {self.n_updates}\n"
            f"  K   = [{self.K[0,0]:+.4f}  {self.K[0,1]:+.4f}]\n"
            f"  K_e = [{Ke[0,0]:+.4f}  {Ke[0,1]:+.4f}]\n"
            f"  W   = [{W[0]:+7.3f}  {W[1]:+7.3f}  {W[2]:+7.3f}\n"
            f"          {W[3]:+7.3f}  {W[4]:+7.3f}  {W[5]:+7.3f}]\n"
            f"  x1  = {self.x[0,0]:+.5f}    x2 = {self.x[1,0]:+.5f}\n"
            f"  K_opt ≈ [-1.1714   0.6825]"
        )


# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = LQR_RL_App()
    win.showMaximized()
    sys.exit(app.exec())

SystemExit: 0

/home/jm-liberty/miniconda3/envs/rl_test/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
